<a href="https://colab.research.google.com/github/rugellioliveira/data-lake-benchmark/blob/main/data_lake_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install pyarrow fastparquet #Biblioteca necessária para lidar com arquivos parquet

In [16]:
import os  # Manipulação de sistema de arquivos (criar pasta, tamanho de arquivo, etc.)
import time  # Medição de tempo de execução (benchmark)
import pandas as pd  # Biblioteca principal para manipulação de dados tabulares

In [17]:
# =========================
# FUNÇÕES UTILITÁRIAS
# =========================

def mb(path):
    """
    Converte o tamanho de um arquivo para megabytes (MB).

    Por padrão, o sistema operacional retorna o tamanho em bytes.
    Aqui convertemos para MB para facilitar comparação entre formatos.

    Fórmula:
    1 MB = 1024 * 1024 bytes
    """
    return os.path.getsize(path) / (1024 * 1024)


def log(msg):
    """
    Função simples de log.

    Em sistemas reais, usaríamos a biblioteca 'logging' para:
    - níveis (INFO, ERROR, DEBUG)
    - logs em arquivo
    - integração com monitoramento (Datadog, CloudWatch, etc.)

    Aqui usamos print para simplicidade didática.
    """
    print(msg)

In [18]:
# =========================
# GERAÇÃO DE DADOS
# =========================

def gerar_dataframe(N):
    """
    Gera um DataFrame com N linhas simulando dados de clientes.

    Esse passo simula a fase de ingestão de dados em um Data Lake,
    onde dados brutos são criados/coletados antes de serem armazenados.

    IMPORTANTE:
    - Tudo é criado em memória (RAM)
    - Para valores muito grandes de N, pode consumir muita memória
    """

    # Lista fixa de cidades (dimensão categórica)
    cidades = ["São Paulo", "Rio de Janeiro", "Curitiba", "Belo Horizonte", "Salvador"]

    return pd.DataFrame({

        # ID sequencial (simula chave primária ou identificador único)
        "id": range(1, N + 1),

        # Nome gerado dinamicamente
        # ⚠️ Cada string ocupa memória → isso escala mal com milhões de linhas
        "nome": [f"Cliente_{i}" for i in range(1, N + 1)],

        # Distribuição de cidades
        # Usamos módulo (%) para criar repetição cíclica
        # Isso evita erro caso N não seja múltiplo do tamanho da lista
        "cidade": [cidades[i % len(cidades)] for i in range(N)],

        # Valores numéricos cíclicos
        # Simula métricas
        # Uso de float para simular dados reais
        "valor": [float(i % 5000) for i in range(N)]
    })

In [19]:
# =========================
# BENCHMARK DE ESCRITA
# =========================

def benchmark_write(df, path, formato):
    """
    Mede o tempo de escrita e o tamanho final do arquivo.

    Esse é um ponto crítico em Data Lakes:
    - custo de armazenamento
    - tempo de ingestão
    """

    # Início do cronômetro (alta precisão)
    t0 = time.perf_counter()

    # =========================
    # CSV
    # =========================
    if formato == "csv":
        """
        CSV (Comma-Separated Values):

        Características:
        - Formato texto
        - Sem compressão
        - Sem schema explícito
        - Muito usado, mas pouco eficiente

        Desvantagens:
        - Arquivos grandes
        - Leitura lenta
        - Tipagem perdida (tudo vira string inicialmente)
        """
        df.to_csv(path, index=False)

    # =========================
    # JSONL
    # =========================
    elif formato == "jsonl":
        """
        JSON Lines (JSONL):

        Cada linha é um JSON independente:
        {"id":1,...}
        {"id":2,...}

        Vantagens:
        - Muito usado em ingestão de dados (streaming)
        - Fácil integração com APIs e logs

        Desvantagens:
        - Verboso (repetição de chaves)
        - Maior tamanho
        - Mais lento para leitura analítica

        IMPORTANTE:
        - 'lines=True' → formato JSONL
        - 'orient=records' → cada linha vira um objeto
        """
        df.to_json(path, orient="records", lines=True, force_ascii=False)

    # =========================
    # PARQUET
    # =========================
    elif formato == "parquet":
        """
        Parquet:

        Formato colunar altamente otimizado.

        Vantagens:
        - Compressão eficiente (snappy)
        - Leitura por coluna (scan seletivo)
        - Muito rápido para analytics
        - Mantém schema e tipos

        'engine=pyarrow':
        - Implementação moderna e performática

        'compression=snappy':
        - Compressão leve e rápida (padrão de mercado)
        """
        df.to_parquet(path, engine="pyarrow", compression="snappy", index=False)

    # Fim do cronômetro
    t1 = time.perf_counter()

    # Log com tempo e tamanho do arquivo
    log(f"[ESCRITA {formato.upper()}] tempo: {t1 - t0:.2f}s | tamanho: {mb(path):.1f} MB")

In [20]:
# =========================
# BENCHMARK DE LEITURA
# =========================

def benchmark_read(path, formato, repeat=3):
    """
    Mede o tempo de leitura dos arquivos.

    'repeat':
    - Executa múltiplas vezes para obter média
    - Reduz impacto de cache do sistema operacional
    """

    tempos = []

    for _ in range(repeat):

        # Início da medição
        t0 = time.perf_counter()

        # =========================
        # LEITURA CSV
        # =========================
        if formato == "csv":
            """
            CSV:
            - Precisa inferir tipos
            - Leitura linha a linha
            - Geralmente mais lento
            """
            _ = pd.read_csv(path)

        # =========================
        # LEITURA JSONL
        # =========================
        elif formato == "jsonl":
            """
            JSONL:
            - Parsing de JSON (mais pesado)
            - Necessário 'lines=True'
            """
            _ = pd.read_json(path, lines=True)

        # =========================
        # LEITURA PARQUET
        # =========================
        elif formato == "parquet":
            """
            Parquet:
            - Leitura colunar otimizada
            - Pode ler apenas colunas específicas (grande vantagem)
            - Geralmente o mais rápido
            """
            _ = pd.read_parquet(path)

        # Fim da medição
        t1 = time.perf_counter()

        tempos.append(t1 - t0)

    # Média dos tempos (benchmark mais confiável)
    tempo_medio = sum(tempos) / len(tempos)

    log(f"[LEITURA {formato.upper()}] tempo médio ({repeat}x): {tempo_medio:.2f}s")

In [21]:
# =========================
# FUNÇÃO PRINCIPAL
# =========================

def main():
    """
    Orquestra todo o fluxo:

    1. Cria estrutura do data lake (pasta)
    2. Gera dados
    3. Salva em diferentes formatos
    4. Mede performance de leitura
    """

    # Cria diretório local simulando um Data Lake (ex: S3, GCS)
    os.makedirs("data_lake", exist_ok=True)

    # Número de linhas (escala do experimento)
    # ⚠️ Ajuste conforme memória disponível
    N = 1_000_000

    # Caminhos dos arquivos
    csv_path = "data_lake/clientes.csv"
    jsonl_path = "data_lake/clientes.jsonl"
    parquet_path = "data_lake/clientes.parquet"

    log("=== DATA LAKE BENCHMARK ===")
    log(f"Gerando dataset com {N:,} linhas...\n")

    # =========================
    # GERAÇÃO
    # =========================
    t0 = time.perf_counter()
    df = gerar_dataframe(N)
    t1 = time.perf_counter()

    log(f"[GERAÇÃO] tempo: {t1 - t0:.2f}s\n")

    # =========================
    # ESCRITA
    # =========================
    benchmark_write(df, csv_path, "csv")
    benchmark_write(df, jsonl_path, "jsonl")
    benchmark_write(df, parquet_path, "parquet")

    print()

    # =========================
    # LEITURA
    # =========================
    benchmark_read(csv_path, "csv")
    benchmark_read(jsonl_path, "jsonl")
    benchmark_read(parquet_path, "parquet")

    log("\n=== FIM ===")

In [22]:
# =========================
# ENTRYPOINT
# =========================

if __name__ == "__main__":
    """
    Ponto de entrada do script.

    Garante que o código só execute quando rodado diretamente,
    e não quando importado como módulo.
    """
    main()

=== DATA LAKE BENCHMARK ===
Gerando dataset com 1,000,000 linhas...

[GERAÇÃO] tempo: 0.87s

[ESCRITA CSV] tempo: 5.33s | tamanho: 38.5 MB
[ESCRITA JSONL] tempo: 1.89s | tamanho: 71.9 MB
[ESCRITA PARQUET] tempo: 0.43s | tamanho: 9.7 MB

[LEITURA CSV] tempo médio (3x): 1.22s
[LEITURA JSONL] tempo médio (3x): 3.09s
[LEITURA PARQUET] tempo médio (3x): 0.55s

=== FIM ===
